In [3]:
#Verify ChromaDB and Sentence Transformers
import chromadb
from sentence_transformers import SentenceTransformer

print("ChromaDB installed")
print("Sentence Transformers installed")

c:\Users\Hima Bindu N\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChromaDB installed
Sentence Transformers installed


In [2]:
!pip install chromadb

  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
    --------------------------------------- 0.5/23.5 MB 932.9 kB/s eta 0:00:25
   - -------------------------------------- 0.8/23.5 MB 1.0 MB/s eta 0:00:23
   - -------------------------------------- 1.0/23.5 MB 1.1 MB/s eta 0:00:21
   -- ------------------------------------- 1.3/23.5 MB 1.1 MB/s eta 0:00:21
   -- ------------------------------------- 1.6/23.5 MB 1.1 MB/s eta 0:00:21
   --- ------------------------------------ 1.8/23.5 MB 1.1 MB/s eta 0:00:20
   --- ------------------------------------ 2.1/23.5 MB 1.1 MB/s eta 0:00:19
   ---- ----------------------------------- 2.4/23.5 MB 1.2 MB/s eta 0:00:19
   ---- ----------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
combined_df["clean_text"] = combined_df["clean_text"].fillna("")

combined_df["clean_text"] = combined_df["clean_text"].astype(str)

In [14]:
print(combined_df["clean_text"].dtype)

str


In [15]:
#Generate Embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    combined_df["clean_text"].tolist(),
    show_progress_bar=True
)

print(embeddings.shape)

Batches: 100%|██████████| 10/10 [00:05<00:00,  1.85it/s]

(314, 384)


In [10]:
import pandas as pd

combined_df = pd.read_csv("processed_dhl_dataset.csv")

print(combined_df.shape)
print(combined_df.columns)

(314, 5)
Index(['title', 'content', 'url', 'source_name', 'clean_text'], dtype='str')


In [16]:
#Create ChromaDB Repository
import chromadb

client = chromadb.PersistentClient(
    path="dhl_chromadb"
)

collection = client.get_or_create_collection(
    name="dhl_intelligence"
)

In [17]:
#Store Documents
for idx, row in combined_df.iterrows():

    collection.add(
        ids=[str(idx)],
        documents=[row["clean_text"]],
        metadatas=[{
            "title": row["title"],
            "source": row["source_name"]
        }]
    )

print("All documents stored successfully")

C:\Users\Hima Bindu N\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:27<00:00, 955kiB/s] 


All documents stored successfully


In [18]:
#Test Retrieval
results = collection.query(
    query_texts=[
        "renewable energy logistics opportunities"
    ],
    n_results=5
)

print(results["metadatas"])

[[{'title': 'DHL Logistics of Things | Insights into the world of logistics', 'source': 'DDGS'}, {'title': 'The Hidden Infrastructure Challenge Behind Renewable Energy Growth', 'source': 'NewsAPI'}, {'title': 'DHL Targets 5x Increase in Energy Transition-Related Revenues to €3 ...', 'source': 'DDGS'}, {'title': 'DHL targets €3bn new energy logistics revenue by 2030', 'source': 'DDGS'}, {'title': 'DHL News Monitoring Service & Press Release Distribution - Shipping ...', 'source': 'DDGS'}]]
